# 🥉 Camada Bronze 

A **camada Bronze** é a base do *Data Lakehouse*, responsável por **armazenar os dados brutos exatamente como foram recebidos** das fontes originais, **sem transformações** que alterem seu conteúdo.  
Ela garante **rastreabilidade e reprocessamento**, funcionando como o “registro histórico imutável” dos dados.

### ✅ Ingestão dos 9 arquivos CSV
A Bronze recebe os arquivos ou streams diretamente das fontes originais (APIs, CSVs, bancos, logs).  
Nenhum dado é modificado, apenas armazenado.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import requests
import json

# definir caminho de landing
landing_path = "/Volumes/medalhao/default/landing/"

# nome do catálogo e banco
catalogo = "medalhao"
bronze_db_name = "bronze"

# função de transformar os csv em tabela que eu trouxe do material das aulas
def ingest_csv(nome_arquivo, nome_tabela):
   
    try:
        table_name = nome_tabela
        landing_path = f"/Volumes/medalhao/default/landing/{nome_arquivo}"

        # Leitura do arquivo CSV
        df = spark.read.csv(landing_path, header=True, inferSchema=True)

        # Validação: arquivo vazio
        if df.count() == 0:
            raise ValueError(f"O arquivo {nome_arquivo} está vazio ou não pôde ser lido.")

        # Adiciona timestamp de ingestão
        df_with_metadata = df.withColumn("ingestion_timestamp", F.current_timestamp())

        # Escrita no formato Delta
        df_with_metadata.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{bronze_db_name}.{table_name}")
        
    except Exception as e:
        print(f"Erro ao processar {nome_tabela}: {str(e)}")


In [0]:
# Função para ingestão de arquivos CSV para as tabelas correspondentes
ingest_csv("olist_customers_dataset.csv", "ft_consumidores")
ingest_csv("olist_geolocation_dataset.csv", "ft_geolocalizacao")
ingest_csv("olist_order_items_dataset.csv", "ft_itens_pedidos")
ingest_csv("olist_order_payments_dataset.csv", "ft_pagamentos_pedidos")
ingest_csv("olist_order_reviews_dataset.csv", "ft_avaliacoes_pedidos")
ingest_csv("olist_orders_dataset.csv", "ft_pedidos")
ingest_csv("olist_products_dataset.csv", "ft_produtos")
ingest_csv("olist_sellers_dataset.csv", "ft_vendedores")
ingest_csv("product_category_name_translation.csv", "dm_categoria_produtos_traducao")

print("Ingestão da camada Bronze concluída!")


### ✅ Ingestão da API da cotação do dólar
A Bronze recebe os arquivos ou streams diretamente das fontes originais (APIs, CSVs, bancos, logs).  
Nenhum dado é modificado, apenas armazenado.

In [0]:
data_inicio_formatada = "01-01-2017"   
data_fim_formatada = "01-01-2020"      


# url fornecida na atividade
url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(" \
      f"dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)" \
      f"?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'" \
      f"&$select=dataHoraCotacao,cotacaoCompra&$format=json"

# fazendo a requisição 
response = requests.get(url)

data_json = response.json()

# converte o JSON para DataFrame Spark
df = spark.createDataFrame(data_json["value"])

# adicionando coluna da fonte dos dados
df_with_source = df.withColumn("source", F.lit("BCB_PTAX_API"))
# adicionando coluna do tempo da ingestão
df_with_metadata = df_with_source.withColumn("ingestion_timestamp", F.current_timestamp())

tabela_destino = f"{catalogo}.{bronze_db_name}.dm_cotacao_dolar"


df_with_metadata.write.format("delta").mode("overwrite").saveAsTable(tabela_destino)


In [0]:
# caso haja necessidade de exclusão das tabelas

''' spark.sql("DROP TABLE IF EXISTS medalhao.bronze.dm_categoria_produtos_traducao")
spark.sql("DROP TABLE IF EXISTS medalhao.bronze.dm_cotacao_dolar")
spark.sql("DROP TABLE IF EXISTS medalhao.bronze.ft_avaliacoes_pedidos")
spark.sql("DROP TABLE IF EXISTS medalhao.bronze.ft_consumidores")
spark.sql("DROP TABLE IF EXISTS medalhao.bronze.ft_geolocalizacao")
spark.sql("DROP TABLE IF EXISTS medalhao.bronze.ft_itens_pedidos")
spark.sql("DROP TABLE IF EXISTS medalhao.bronze.ft_pagamentos_pedidos")
spark.sql("DROP TABLE IF EXISTS medalhao.bronze.ft_pedidos")
spark.sql("DROP TABLE IF EXISTS medalhao.bronze.ft_produtos")
spark.sql("DROP TABLE IF EXISTS medalhao.bronze.ft_vendedores") '''


